# MCP + OpenRouter LLM — Complete Local Jupyter Demo

This notebook demonstrates two MCP integrations:

## Part A — Your own MCP server

```text
Terminal 1                         Jupyter Notebook
Custom FastMCP server              MCP client + OpenRouter LLM
        │                                      │
        └── http://127.0.0.1:8000/mcp ─────────┘
```

## Part B — A free remote MCP server

```text
Jupyter Notebook
MCP client + OpenRouter LLM
        │
        │ HTTPS
        ▼
Microsoft Learn MCP Server
https://learn.microsoft.com/api/mcp
```

The Microsoft Learn MCP Server is publicly available, requires no authentication, and provides current Microsoft documentation through MCP.


# Project files

Keep these files in the same folder:

```text
mcp_final_complete/
├── custom_mcp_http_server.py
└── MCP_OpenRouter_Complete_Local_Jupyter.ipynb
```

Before running Part A, start the custom server in a separate terminal:

```bash
source mcp_env/bin/activate
python custom_mcp_http_server.py
```

Keep that terminal open.


## 1. Install the libraries

In [ ]:
# pip install "mcp[cli]"
# for the server running

In [ ]:
%pip install -q -U "mcp[cli]>=1.20,<2" openai

Restart the Jupyter kernel after installation if the imports fail.

## 2. Configure OpenRouter

In [1]:
import os
from getpass import getpass

from openai import OpenAI


os.environ["OPENROUTER_API_KEY"] = getpass(
    "Enter your OpenRouter API key: "
)

MODEL_NAME = "openai/gpt-4o-mini"

llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

print(f"OpenRouter client configured with: {MODEL_NAME}")


OpenRouter client configured with: openai/gpt-4o-mini


## 3. Import MCP classes

In [2]:
import json

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


## 4. Helper functions

The MCP server and the LLM use slightly different tool-schema formats.

These helpers:

1. convert MCP tools into OpenAI-compatible tools;
2. convert MCP results into text;
3. allow the LLM to choose and call MCP tools.


In [3]:
def mcp_tools_to_llm_tools(mcp_tools):
    """Convert MCP tool schemas to OpenAI-compatible schemas."""

    return [
        {
            "type": "function",
            "function": {
                "name": tool.name,
                "description": tool.description or "",
                "parameters": tool.inputSchema,
            },
        }
        for tool in mcp_tools
    ]


def mcp_result_to_text(result):
    """Extract readable text from an MCP tool result."""

    values = []

    for item in result.content:
        text = getattr(item, "text", None)
        values.append(text if text is not None else str(item))

    return "\n".join(values)


In [4]:
async def ask_llm_using_mcp(session, user_question, system_message="Use available tools when relevant.", max_tool_rounds=3):
    """Let the LLM discover and call tools from an MCP session."""

    # Ask the MCP server which tools it provides.
    tools_response = await session.list_tools()

    print("Available MCP tools:")
    for tool in tools_response.tools:
        print(f"- {tool.name}: {tool.description}")

    # Convert MCP tools to the format expected by OpenRouter.
    llm_tools = mcp_tools_to_llm_tools(
        tools_response.tools
    )

    print("LLM Tools", llm_tools)

    messages = [
        {
            "role": "system",
            "content": system_message,
        },
        {
            "role": "user",
            "content": user_question,
        },
    ]

    # The LLM may need more than one tool round.
    for _ in range(max_tool_rounds):

        response = llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            tools=llm_tools,
            tool_choice="auto",
            temperature=0.2,
            max_tokens=150,
        )

        assistant_message = response.choices[0].message

        # No tool call means the model has produced its final answer.
        if not assistant_message.tool_calls:
            return assistant_message.content

        # Save the assistant's tool request in conversation history.
        messages.append(
            assistant_message.model_dump(exclude_none=True)
        )

        # Execute all tools requested in this round.
        for tool_call in assistant_message.tool_calls:

            tool_name = tool_call.function.name

            try:
                tool_arguments = json.loads(
                    tool_call.function.arguments or "{}"
                )
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid tool arguments from LLM: {exc}"
                ) from exc

            print(f"\nLLM selected tool: {tool_name}")
            print("Arguments:", tool_arguments)

            # Call the tool through the MCP session.
            result = await session.call_tool(
                tool_name,
                arguments=tool_arguments,
            )

            result_text = mcp_result_to_text(result)

            print("MCP result preview:")
            print(result_text[:2000])

            # Return the MCP result to the LLM.
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_name,
                    "content": result_text,
                }
            )

    return (
        "The maximum number of MCP tool rounds was reached "
        "before the model produced a final answer."
    )


# Part A — Custom FastMCP server

Before continuing, open a terminal in the project folder and run:

```bash
python custom_mcp_http_server.py
```

The server should listen at:

```text
http://127.0.0.1:8000/mcp
```


## 5. Test the custom server connection

In [5]:
CUSTOM_MCP_URL = "http://127.0.0.1:8000/mcp"


async def test_custom_server():
    async with streamable_http_client(CUSTOM_MCP_URL) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            await session.initialize()

            print("Connected to custom MCP server.")

            tools = await session.list_tools()

            print("\nAvailable tools:")
            for tool in tools.tools:
                print("-", tool.name)
            print("\nAvailable tools description:")
            for tool in tools.tools:
                print("-", tool.description)
            print("\nAvailable tools schema:")
            for tool in tools.tools:
                print("-", tool.inputSchema)


await test_custom_server()


  + Exception Group Traceback (most recent call last):
  |   File "d:\GitHub\Backend-Development-AI\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\Mohankumar MC\AppData\Local\Temp\ipykernel_24688\3026361730.py", line 31, in <module>
  |     await test_custom_server()
  |   File "C:\Users\Mohankumar MC\AppData\Local\Temp\ipykernel_24688\3026361730.py", line 5, in test_custom_server
  |     async with streamable_http_client(CUSTOM_MCP_URL) as (
  |                ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  |   File "C:\Python314\Lib\contextlib.py", line 235, in __aexit__
  |     await self.gen.athrow(value)
  |   File "d:\GitHub\Backend-Development-AI\venv\Lib\site-packages\mcp\client\streamable_http.py", line 647, in streamable_http_client
  |     async with anyio.create_task_group() as tg:
  |                ~~~~~~~~~~~~~~~~~~~~~~~^^
  |   File "d:\GitHub\Backend-Developm

## 6. Let the OpenRouter LLM use the custom MCP tools

In [6]:
async def ask_custom_mcp(question):
    async with streamable_http_client(CUSTOM_MCP_URL) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            await session.initialize()

            return await ask_llm_using_mcp(
                session=session,
                user_question=question,
                system_message=(
                    "You are a helpful travel-budget assistant. "
                    "Use the MCP calculation tools instead of "
                    "calculating manually."
                ),
            )


In [7]:
custom_answer = await ask_custom_mcp(
    "I am travelling for 3 days. "
    "My daily budget is 2500 and my fixed cost is 8000. "
    "What is my estimated total budget?"
)

print("\nFinal answer:")
print(custom_answer)


  + Exception Group Traceback (most recent call last):
  |   File "d:\GitHub\Backend-Development-AI\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\Mohankumar MC\AppData\Local\Temp\ipykernel_24688\1724215771.py", line 1, in <module>
  |     custom_answer = await ask_custom_mcp(
  |                     ^^^^^^^^^^^^^^^^^^^^^
  |     ...<3 lines>...
  |     )
  |     ^
  |   File "C:\Users\Mohankumar MC\AppData\Local\Temp\ipykernel_24688\496022282.py", line 2, in ask_custom_mcp
  |     async with streamable_http_client(CUSTOM_MCP_URL) as (
  |                ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  |   File "C:\Python314\Lib\contextlib.py", line 235, in __aexit__
  |     await self.gen.athrow(value)
  |   File "d:\GitHub\Backend-Development-AI\venv\Lib\site-packages\mcp\client\streamable_http.py", line 647, in streamable_http_client
  |     async with anyio.create_task_gro

## 7. Read a resource from the custom server

In [9]:
async with streamable_http_client(CUSTOM_MCP_URL) as (
    read_stream,
    write_stream,
    _,
):
    async with ClientSession(
        read_stream,
        write_stream,
    ) as session:
        await session.initialize()

        resources = await session.list_resources()

        print("Available resources:")
        for resource in resources.resources:
            print("-", resource.uri)

        result = await session.read_resource(
            "course://mcp/introduction"
        )

        print("\nResource content:")
        for content in result.contents:
            print(getattr(content, "text", content))


  + Exception Group Traceback (most recent call last):
  |   File "d:\GitHub\Backend-Development-AI\venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\Mohankumar MC\AppData\Local\Temp\ipykernel_24688\701157966.py", line 1, in <module>
  |     async with streamable_http_client(CUSTOM_MCP_URL) as (
  |                ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  |   File "C:\Python314\Lib\contextlib.py", line 235, in __aexit__
  |     await self.gen.athrow(value)
  |   File "d:\GitHub\Backend-Development-AI\venv\Lib\site-packages\mcp\client\streamable_http.py", line 647, in streamable_http_client
  |     async with anyio.create_task_group() as tg:
  |                ~~~~~~~~~~~~~~~~~~~~~~~^^
  |   File "d:\GitHub\Backend-Development-AI\venv\Lib\site-packages\anyio\_backends\_asyncio.py", line 815, in __aexit__
  |     raise BaseExceptionGroup(
  |         "unhandled errors in a T

# Part B — Free remote Microsoft Learn MCP Server

The Microsoft Learn MCP Server:

- is hosted remotely by Microsoft;
- uses Streamable HTTP;
- requires no authentication;
- is free to use;
- searches official Microsoft documentation.

Endpoint:

```text
https://learn.microsoft.com/api/mcp
```

You do not start this server yourself.


## 8. Test the free remote MCP connection

In [8]:
MICROSOFT_LEARN_MCP_URL = (
    "https://learn.microsoft.com/api/mcp"
)


async def test_microsoft_learn_mcp():
    async with streamable_http_client(
        MICROSOFT_LEARN_MCP_URL
    ) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            await session.initialize()

            print(
                "Connected to Microsoft Learn MCP server."
            )

            tools = await session.list_tools()

            print("\nAvailable tools:")
            for tool in tools.tools:
                print(f"- {tool.name}: {tool.description}")
                print("****"*30)


await test_microsoft_learn_mcp()


Connected to Microsoft Learn MCP server.

Available tools:
- microsoft_docs_search: Search official Microsoft/Azure documentation to find the most relevant and trustworthy content for a user's query. This tool returns up to 10 high-quality content chunks (each max 500 tokens), extracted from Microsoft Learn and other official sources. Each result includes the article title, URL, and a self-contained content excerpt optimized for fast retrieval and reasoning. Always use this tool to quickly ground your answers in accurate, first-party Microsoft/Azure knowledge.

## Follow-up Pattern
To ensure completeness, use microsoft_docs_fetch when high-value pages are identified by search. The fetch tool complements search by providing the full detail. This is a required step for comprehensive results.
************************************************************************************************************************
- microsoft_code_sample_search: Search for code snippets and examples in offic

## 9. Let the OpenRouter LLM use Microsoft Learn MCP

In [10]:
async def ask_microsoft_learn_mcp(question):
    async with streamable_http_client(
        MICROSOFT_LEARN_MCP_URL
    ) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            await session.initialize()

            return await ask_llm_using_mcp(
                session=session,
                user_question=question,
                system_message=(
                    "You are a technical instructor. "
                    "Use Microsoft Learn MCP tools to search "
                    "official Microsoft documentation. "
                    "Base the answer only on returned tool results. "
                    "Keep the explanation beginner-friendly."
                ),
            )


In [11]:
learn_answer = await ask_microsoft_learn_mcp(
    "Using official Microsoft documentation, "
    "explain Azure Functions in five simple points."
)

print("\nFinal answer:")
print(learn_answer)


Available MCP tools:
- microsoft_docs_search: Search official Microsoft/Azure documentation to find the most relevant and trustworthy content for a user's query. This tool returns up to 10 high-quality content chunks (each max 500 tokens), extracted from Microsoft Learn and other official sources. Each result includes the article title, URL, and a self-contained content excerpt optimized for fast retrieval and reasoning. Always use this tool to quickly ground your answers in accurate, first-party Microsoft/Azure knowledge.

## Follow-up Pattern
To ensure completeness, use microsoft_docs_fetch when high-value pages are identified by search. The fetch tool complements search by providing the full detail. This is a required step for comprehensive results.
- microsoft_code_sample_search: Search for code snippets and examples in official Microsoft Learn documentation. This tool retrieves relevant code samples from Microsoft documentation pages providing developers with practical implementat

## 10. Try another Microsoft documentation question

In [12]:
learn_answer = await ask_microsoft_learn_mcp(
    "Find official Microsoft documentation about creating "
    "a Python Azure Function and summarise the main steps."
)

print("\nFinal answer:")
print(learn_answer)


Available MCP tools:
- microsoft_docs_search: Search official Microsoft/Azure documentation to find the most relevant and trustworthy content for a user's query. This tool returns up to 10 high-quality content chunks (each max 500 tokens), extracted from Microsoft Learn and other official sources. Each result includes the article title, URL, and a self-contained content excerpt optimized for fast retrieval and reasoning. Always use this tool to quickly ground your answers in accurate, first-party Microsoft/Azure knowledge.

## Follow-up Pattern
To ensure completeness, use microsoft_docs_fetch when high-value pages are identified by search. The fetch tool complements search by providing the full detail. This is a required step for comprehensive results.
- microsoft_code_sample_search: Search for code snippets and examples in official Microsoft Learn documentation. This tool retrieves relevant code samples from Microsoft documentation pages providing developers with practical implementat

# Complete flow

```text
User asks a question
        ↓
Python connects to an MCP server
        ↓
ClientSession performs the MCP handshake
        ↓
Python requests list_tools()
        ↓
MCP tool schemas are converted for OpenRouter
        ↓
LLM selects a tool and generates arguments
        ↓
Python calls session.call_tool()
        ↓
MCP server executes the tool
        ↓
Tool result returns to Python
        ↓
Python sends the result back to the LLM
        ↓
LLM generates the final human-readable answer
```

## Part A versus Part B

| Feature | Part A | Part B |
|---|---|---|
| Server owner | You | Microsoft |
| Location | Your computer | Remote cloud server |
| URL | `127.0.0.1:8000/mcp` | `learn.microsoft.com/api/mcp` |
| Must you start it? | Yes | No |
| Transport | Streamable HTTP | Streamable HTTP |
| Main purpose | Demonstrate your own tools | Search official Microsoft documentation |
